In [ ]:
import pandas as pd
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Two-tier Feature MRMD-RF-RFE

In [ ]:
import numpy as np
import pandas as pd

from sklearn.feature_selection import mutual_info_classif, RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# -------------------------
# Encode binary labels
# -------------------------
def make_binary(y):
    if y.dtype.kind in "biufc":
        uniq = np.unique(y)
        if len(uniq) != 2:
            raise ValueError("Target is not binary")
        return LabelEncoder().fit_transform(y)
    else:
        return LabelEncoder().fit_transform(y.astype(str))

# -------------------------
# MRMD feature ranking
# -------------------------
def mrmd(X, y):
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.fillna(X.median())

    relevance = mutual_info_classif(X.values, y, random_state=42)
    relevance = pd.Series(relevance, index=X.columns)

    corr = X.corr().abs()
    redundancy = (corr.sum(axis=1) - 1) / (len(X.columns) - 1)
    redundancy = redundancy.fillna(0)

    score = relevance - redundancy

    ranking = pd.DataFrame({
        "feature": X.columns,
        "mrmd_score": score
    }).sort_values("mrmd_score", ascending=False)

    return ranking

# -------------------------
# RF-RFE selection
# -------------------------
def rf_rfe(X, y, n_features):
    rf = RandomForestClassifier(
        n_estimators=500,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    )
    selector = RFE(rf, n_features_to_select=n_features)
    selector.fit(X, y)

    return X.columns[selector.support_]

# -------------------------
# MAIN PIPELINE
# -------------------------
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/TPpred/Test DS/AVP Test DS Hyb.csv")


X = df.iloc[:, :-1]
y = df.iloc[:, -1]

y = make_binary(y)

# -------- Pass 1: MRMD --------
mrmd_rank = mrmd(X, y)
mrmd_rank.to_csv("MRMD_ranking.csv", index=False)

K = 820   # number of features after MRMD
mrmd_features = mrmd_rank["feature"].head(K).tolist()

df_pass1 = df[mrmd_features + [df.columns[-1]]]
df_pass1.to_csv("selected_pass1_MRMD.csv", index=False)

# -------- Pass 2: RF-RFE --------
M = 320    # final number of features
final_features = rf_rfe(X[mrmd_features], y, M);

df_final = df[list(final_features) + [df.columns[-1]]]
df_final.to_csv("/content/drive/MyDrive/Colab Notebooks/TPpred/AVP_selected_pass-TestDS.csv", index=False)

print("✅ Feature selection completed")
print("Final selected features:")
for f in final_features:
    print("-", f)


✅ Feature selection completed
Final selected features:
- 0.30556.2
- 6.5564
- -2.5748
- -23.253
- -17.377
- -22.021
- -19.156
- 2.5085
- -5.1771917
- 0.27778.1
- -7.504
- 4.5661
- 67.551
- 12.985
- -33.057
- -66.645
- -16.929
- 0.2245
- -10.835
- 0.26036084
- 1.33
- 31.113
- 1.5994
- 7.593
- 8.1401
- 29.816
- 5.878
- -18.314
- -45.836
- 0.21611899
- -0.15152
- 0.055556.6
- -1.0884
- 0.30556.3
- -0.48047
- -143.34
- 6.9528
- -2.3654
- 15.353
- 0.37143
- 7.9788
- -102.92
- -12.98
- -28.05
- 10.183
- 0.55556
- 0.083333
- 3.9631
- -14.626
- 39.61
- -47.767
- -49.146
- -0.14453836
- -11.227
- -37.279
- -0.049799785
- 0.3
- 45.432
- 4.7771
- -33.511
- -1.3636
- -0.036575172
- -18.94
- -47.773
- -49.095
- 6.1462
- 0.1446298
- 26.662
- 0.69444
- 0.19444.3
- 83.436
- 52.125
- -12.216
- 0.040989254
- -36.593
- 26.093
- 0.083333.5
- 67.705
- 358.24
- -0.045125645
- 0.24675032
- 0.08355296
- -64.397
- -0.021422202
- -125.64
- 0.008007563
- -13.96
- 0.053727973
- -39.043
- 275.22
- -72.302
- -0.033